In [ ]:
# Install Conda on Colab
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:08
🔁 Restarting kernel...


In [ ]:
# 1. Clean up
!conda env remove -n miracl_env -y || true

# 2. Create Base Env (MKL 2021 + Compilers)
print("--- Creating Environment ---")
!conda create -n miracl_env -y \
    python=3.8 \
    openjdk=11 \
    maven \
    mkl=2021.4.0 \
    gxx_linux-64 \
    gcc_linux-64 \
    sysroot_linux-64=2.17 \
    -c conda-forge

# 3. Install Binaries (Faiss, Torch, PyTrec_Eval) via Conda
# Installing pytrec_eval via conda bypasses the pip build failure
print("\n--- Installing Binaries ---")
!conda install -n miracl_env -y -c pytorch -c nvidia -c conda-forge \
    faiss-gpu cudatoolkit=11.8 pytorch \
    pytrec_eval

# 4. Install Pyserini (Direct Wheel)
print("\n--- Installing Pyserini ---")
!conda run -n miracl_env pip install pyserini==0.19.0 --no-deps

# 5. Install Remaining Python Libs (Pip)
print("\n--- Installing Python Libs ---")
# Removed pytrec_eval from here since we installed it via conda above
!conda run -n miracl_env pip install \
    numpy==1.23.5 \
    cython \
    nmslib==2.1.1 \
    spacy==3.5.3 \
    lightgbm==3.3.5 \
    onnxruntime==1.15.1 \
    pyjnius==1.6.1 \
    transformers==4.30.0 \
    sentencepiece \
    pandas \
    scikit-learn \
    tqdm \
    protobuf==3.20.0


Remove all packages in environment /usr/local/envs/miracl_env:


## Package Plan ##

  environment location: /usr/local/envs/miracl_env


The following packages will be REMOVED:

  _openmp_mutex-4.5-7_kmp_llvm
  alsa-lib-1.2.15.2-hb03c661_0
  binutils_impl_linux-64-2.45-default_hfdba357_105
  binutils_linux-64-2.45-default_h4852527_105
  blas-2.112-mkl
  blas-devel-3.9.0-12_linux64_mkl
  bzip2-1.0.8-hda65f42_8
  ca-certificates-2026.1.4-hbd8a1cb_0
  cairo-1.18.4-he90730b_1
  cpython-3.8.20-py38hd8ed1ab_2
  cuda-cudart-12.4.127-0
  cuda-cupti-12.4.127-0
  cuda-libraries-12.4.1-0
  cuda-nvrtc-12.4.127-0
  cuda-nvtx-12.4.127-0
  cuda-opencl-12.4.127-0
  cuda-runtime-12.4.1-0
  cudatoolkit-11.8.0-h4ba93d1_13
  faiss-1.8.0-py38cuda118h1516ac4_1_cuda
  faiss-gpu-1.8.0-h0240f8b_2
  filelock-3.16.1-pyhd8ed1ab_0
  font-ttf-dejavu-sans-mono-2.37-hab24e00_0
  font-ttf-inconsolata-3.000-h77eed37_0
  font-ttf-source-code-pro-2.038-h77eed37_0
  font-ttf-ubuntu-0.83-h77eed37_3
  fontconfig-2.15.0-h7

In [ ]:
print("--- REPAIRING PYTORCH ---")

# 1. Install PyTorch (compatible with Python 3.8 and our Faiss version)
# We re-assert faiss-gpu to ensure Conda keeps them compatible
!conda install -n miracl_env -c pytorch -c nvidia pytorch faiss-gpu=1.7.4 cuda-version=11.8 -y

# 2. Critical Check: Verify ALL Imports
print("\n--- FINAL SYSTEM CHECK ---")
!env LD_LIBRARY_PATH=/usr/local/envs/miracl_env/lib \
 conda run -n miracl_env python -c "import faiss; import torch; print(f'✅ SUCCESS: Faiss {faiss.__version__} | Torch {torch.__version__}')"

--- REPAIRING PYTORCH ---
Channels:
 - pytorch
 - nvidia
 - conda-forge
Platform: linux-64
Solving environment: - \ | / - done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 25.11.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/miracl_env

  added / updated specs:
    - cuda-version=11.8
    - faiss-gpu=1.7.4
    - pytorch


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    cuda-version-11.8          |       h70ddcb2_3          21 KB  conda-forge
    faiss-1.7.4                |py38cuda112h48d0473_0_cuda         1.4 MB  conda-forge
    faiss-gpu-1.7.4            |       h788eb59_0          19 KB  conda-forge
    libfaiss-1.7.4             |cuda112hb18a002_0_cuda        68.0 MB  conda-forge
    libfaiss-avx2-1.7.4        |cuda112h1

In [ ]:
# Find the java binary inside the conda environment
!find /usr/local/envs/miracl_env -name java | grep bin/java

/usr/local/envs/miracl_env/lib/jvm/bin/java
/usr/local/envs/miracl_env/bin/java


In [ ]:
# This should print OpenJDK 11
!conda run -n miracl_env java -version

openjdk version "11.0.29-internal" 2025-10-21
OpenJDK Runtime Environment (build 11.0.29-internal+0-adhoc..src)
OpenJDK 64-Bit Server VM (build 11.0.29-internal+0-adhoc..src, mixed mode)



In [ ]:
%%writefile run_bm25.py
import sys
import os
import subprocess
import glob

# --- ROBUST JAVA SETUP ---
def setup_java():
    print("🔍 Auto-detecting Java environment...")
    try:
        # 1. Find Java Binary in Conda
        # This returns something like /usr/local/envs/miracl_env/bin/java
        java_bin = subprocess.check_output("which java", shell=True).decode("utf-8").strip()

        # 2. Determine JAVA_HOME (Up one level from bin)
        java_home = os.path.dirname(os.path.dirname(java_bin))
        os.environ["JAVA_HOME"] = java_home
        print(f"   ✅ JAVA_HOME: {java_home}")

        # 3. Find libjvm.so (Critical for Pyjnius)
        # It's usually in lib/server/libjvm.so, but path varies by OS/Version
        print("   🔍 Searching for libjvm.so...")
        jvm_search = glob.glob(f"{java_home}/**/libjvm.so", recursive=True)

        if jvm_search:
            libjvm_path = jvm_search[0]
            os.environ["JVM_PATH"] = libjvm_path
            print(f"   ✅ JVM_PATH:  {libjvm_path}")
        else:
            print("   ⚠️ WARNING: Could not find libjvm.so. Pyjnius might fail.")

    except Exception as e:
        print(f"   ❌ Java Setup Error: {e}")

# Run setup BEFORE importing pyserini
setup_java()
# -------------------------

from pyserini.search.lucene import LuceneSearcher
from pyserini.search import get_topics, get_qrels
import pytrec_eval
from tqdm import tqdm

def main():
    print("\n--- 🚀 STARTING BM25 BASELINE ---")

    # 1. Load Index
    print("Loading Index 'miracl-v1.0-ar'...")
    try:
        searcher = LuceneSearcher.from_prebuilt_index('miracl-v1.0-ar')
        searcher.set_language('arabic')
        # Explicit parameters from official reproduction guide
        searcher.set_bm25(k1=0.9, b=0.4)
    except Exception as e:
        print(f"❌ Error loading index: {e}")
        sys.exit(1)

    # 2. Load Queries
    topics = get_topics('miracl-v1.0-ar-dev')
    print(f"Loaded {len(topics)} queries.")

    # 3. Retrieval
    output_file = "results/baseline/bm25_python.txt"
    os.makedirs("results/baseline", exist_ok=True)

    print("Retrieving (k=100)...")
    with open(output_file, 'w') as f:
        for qid in tqdm(topics.keys(), desc="Search"):
            query_text = topics[qid]['title']
            hits = searcher.search(query_text, k=100)
            for i, hit in enumerate(hits):
                f.write(f"{qid} Q0 {hit.docid} {i+1} {hit.score:.5f} bm25\n")

    # 4. Evaluate
    print("\n--- 📊 EVALUATION ---")
    raw_qrels = get_qrels('miracl-v1.0-ar-dev')
    qrels = {str(q): {str(d): int(s) for d, s in v.items()} for q, v in raw_qrels.items()}

    with open(output_file, 'r') as f:
        run_data = pytrec_eval.parse_run(f)

    evaluator = pytrec_eval.RelevanceEvaluator(qrels, {'recall_100', 'recall_10', 'ndcg_cut_10'})
    results = evaluator.evaluate(run_data)

    aggs = {'recall_100': 0.0, 'recall_10': 0.0, 'ndcg_cut_10': 0.0}
    for q in results:
        for m in aggs:
            aggs[m] += results[q][m]

    print("-" * 40)
    print(f"Recall@100: {aggs['recall_100']/len(results):.4f} (Target: ~0.889)")
    print(f"NDCG@10:    {aggs['ndcg_cut_10']/len(results):.4f} (Target: ~0.481)")
    print("-" * 40)
    print(f"Recall@10:  {aggs['recall_10']/len(results):.4f} (Thesis Baseline)")
    print("-" * 40)

if __name__ == "__main__":
    main()

Overwriting run_bm25.py


In [ ]:
# We use !env LD_LIBRARY_PATH just to keep the MKL fix active
!env LD_LIBRARY_PATH=/usr/local/envs/miracl_env/lib \
 conda run -n miracl_env python -u run_bm25.py

🔍 Auto-detecting Java environment...
   ✅ JAVA_HOME: /usr/local/envs/miracl_env
   🔍 Searching for libjvm.so...
   ✅ JVM_PATH:  /usr/local/envs/miracl_env/lib/jvm/lib/server/libjvm.so

Traceback (most recent call last):
  File "run_bm25.py", line 40, in <module>
    import pytrec_eval
ModuleNotFoundError: No module named 'pytrec_eval'

ERROR conda.cli.main_run:execute(125): `conda run python -u run_bm25.py` failed. (See above for error)
